# 04 — L1 preview (overnight quick-pass, NOT the claims notebook)

Auto-executed by `overnight-interp-0901.sh` after the replay cache lands. Purpose:
give the morning review an honest first look at **H0-3/H0-4/H0-5** (`PREREG-9B.md`)
on Qwen3.5-9B. Formal claims, full controls, and the text baseline live in the
hand-driven `04-l1-probes.ipynb` that follows. Everything here is preview-grade:
grouped CV, shuffled controls, per-lead trial counts — but layer selection is shown,
not yet validation-frozen.

**Morning checklist:** (1) cache completeness table below; (2) the per-lead AUROC
figure — does any layer's curve sit above chance at lead ≥ 1 with the shuffled band
at 0.5?; (3) probe vs logit-report at matched leads; (4) the round-1-leak exclusion
counts; (5) anything weird → open the trial in inspect view before believing a number.


In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

CACHE = "../microscope/cache/qwen35-9b-v1"
npz_files = sorted(glob.glob(os.path.join(CACHE, "*.npz")))
print(f"cache dir: {CACHE}")
print(f"trials cached: {len(npz_files)}")
trials = []
for f in npz_files:
    meta_f = f.replace(".npz", ".json")
    if not os.path.exists(meta_f):
        print("  missing meta:", os.path.basename(f)); continue
    z = np.load(f)
    meta = json.load(open(meta_f))
    trials.append({"resid": z["resid"], "ev": z.get("report_ev"), "meta": meta})
if not trials:
    print("NO CACHE YET — overnight replay has not produced trials; nothing to preview.")
else:
    shapes = {t["resid"].shape for t in trials}
    print("resid shapes:", shapes)
    from collections import Counter
    print(Counter((t["meta"]["persona"], t["meta"]["outcome"]) for t in trials))


## Dataset: per-round vectors, pre-event labels

Label = trial outcome is `leaked` (primary endpoint). For lead-time analysis, round r
of a leaked trial gets lead = leak_round − r; only pre-event rounds (lead ≥ 1) enter.
Non-leak trials contribute all rounds as negatives at every lead (standard
survival-style alignment; the formal notebook will also try event-aligned negatives).
Round-1 leaks have no pre-event window — excluded and counted.


In [ ]:
if trials:
    rows = []          # (trial_idx, round, layer-resid, leaked, lead)
    r1_leaks = 0
    for i, t in enumerate(trials):
        m = t["meta"]; leaked = m["outcome"] in ("leaked", "left_after_leak")
        ev_round = m.get("leak_round")
        n_rounds = t["resid"].shape[0]
        if leaked and ev_round == 1:
            r1_leaks += 1; continue
        for r in range(1, n_rounds + 1):
            lead = (ev_round - r) if (leaked and ev_round) else None
            if leaked and (lead is None or lead < 1):
                continue                      # at/after event: excluded from lead sets
            rows.append((i, r, t["resid"][r - 1], leaked, lead))
    print(f"round-1 leaks excluded: {r1_leaks}")
    print(f"pre-event/negative round-vectors: {len(rows)}")
    n_layers = trials[0]["resid"].shape[1]
    from collections import Counter
    print("lead counts (leaked rows):", Counter(l for *_, lk, l in
          [(a,b,c,d,e) for a,b,c,d,e in rows] if lk and l))


## L1 quick pass — per-layer AUROC, grouped CV, shuffled control

Probe: logistic regression, standardized, GroupKFold(5) by trial. AUROC per layer at
"any pre-event round" first (the coarse H0-3 screen), then per-lead for the best
layers. Shuffled-label control repeated per layer.


In [ ]:
if trials and rows:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold, cross_val_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    X = np.stack([r[2] for r in rows]).astype(np.float32)   # [n, layers, d]
    y = np.array([r[3] for r in rows])
    groups = np.array([r[0] for r in rows])
    leads = np.array([r[4] if r[4] is not None else -1 for r in rows])
    rng = np.random.default_rng(0)

    def layer_auc(Xl, yy, gg):
        pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        cv = GroupKFold(min(5, len(np.unique(gg))))
        return cross_val_score(pipe, Xl, yy, cv=cv, groups=gg,
                               scoring="roc_auc").mean()

    aucs, aucs_shuf = [], []
    for L in range(n_layers):
        aucs.append(layer_auc(X[:, L], y, groups))
        aucs_shuf.append(layer_auc(X[:, L], rng.permutation(y), groups))
    aucs, aucs_shuf = np.array(aucs), np.array(aucs_shuf)
    best = int(np.argmax(aucs))
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(aucs, label="probe"); ax.plot(aucs_shuf, label="shuffled", alpha=0.6)
    ax.axhline(0.5, ls=":", c="gray"); ax.set_xlabel("layer"); ax.set_ylabel("AUROC")
    ax.set_title(f"pre-event leaked-vs-not AUROC by layer (best L{best}={aucs[best]:.2f})")
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"best layer {best}: AUROC {aucs[best]:.3f} (shuffled {aucs_shuf[best]:.3f})")


## Lead-time preview — probe vs logit self-report (H0-4/H0-5 first look)

AUROC restricted to rows at each lead (negatives: all non-leak rows at the same round
distribution). Self-report rival: logit-readout expected values (mean over the 6 items,
and the single best item) at the same rounds. Text baseline is NOT here (formal
notebook); chance band from the shuffled probe.


In [ ]:
if trials and rows:
    from sklearn.metrics import roc_auc_score
    Xb = X[:, best]
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    # out-of-fold scores for honest per-lead AUROC
    from sklearn.model_selection import cross_val_predict
    cv = GroupKFold(min(5, len(np.unique(groups))))
    prob = cross_val_predict(pipe, Xb, y, cv=cv, groups=groups,
                             method="predict_proba")[:, 1]
    ev = []
    for r_ in rows:
        t = trials[r_[0]]; rr = r_[1]
        e = t["ev"]
        ev.append(np.nanmean(e[rr - 1]) if e is not None and rr <= len(e) else np.nan)
    ev = np.array(ev)

    print(f"{'lead':>4} {'n+':>4} {'n-':>5} {'probe':>6} {'report':>7}")
    for lead in range(1, 8):
        sel = (leads == lead) | (~y.astype(bool))
        yy = y[sel]
        if yy.sum() < 3:
            continue
        a_p = roc_auc_score(yy, prob[sel])
        ok = ~np.isnan(ev[sel])
        a_r = roc_auc_score(yy[ok], -ev[sel][ok]) if yy[ok].sum() >= 3 else float("nan")
        print(f"{lead:>4} {int(yy.sum()):>4} {int((~yy.astype(bool)).sum()):>5} "
              f"{a_p:>6.2f} {a_r:>7.2f}")
    print("\nreport column uses -E[v] of mean item (falling resolve should predict "
          "leak); sign convention checked in the formal notebook.")


## Direction + distribution quick look (L2/L3 seeds, mixture check)

Diff-in-means (pre-event leaked minus held rounds) at the best layer; per-round
projection violins by outcome — the dial-vs-switch eyeball from the survey notebook.


In [ ]:
if trials and rows:
    d = X[y.astype(bool), best].mean(0) - X[~y.astype(bool), best].mean(0)
    d /= np.linalg.norm(d)
    proj = Xb @ d
    fig, ax = plt.subplots(figsize=(8, 3))
    rounds_ = np.array([r[1] for r in rows])
    for leaked, c in ((True, "tab:red"), (False, "tab:blue")):
        sel = y.astype(bool) == leaked
        means = [proj[sel & (rounds_ == rr)].mean()
                 for rr in range(1, 9) if (sel & (rounds_ == rr)).sum() > 1]
        ax.plot(range(1, len(means) + 1), means, "o-", color=c,
                label=f"leaked={leaked}")
    ax.set_xlabel("round"); ax.set_ylabel("proj on diff-mean d")
    ax.legend(); ax.set_title("per-round mean projection (pre-event rows only)")
    plt.tight_layout(); plt.show()


## Discussion (to be filled by the morning human)

- H0-3 verdict candidate: …
- Probe vs report at matched leads: …
- Suspicious cells to open in inspect view: …
- Decisions for the formal `04-l1-probes.ipynb`: layer freeze, negative-set choice,
  text-baseline features.
